1. construct the DCOPF latex tutorial
2. check the PU in the formulation, since the 30bus is infeasible

In [6]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
file=".\\old\\DCOPF-main\\DCOPF-main\\excel_outputs\\pglib_opf_case1951_rte.xlsx"
mpc_data = pd.read_excel(file, sheet_name=['baseMVA', 'bus', 'gen', 'gencost', 'branch'])

In [10]:
# Create gurobipy Model
model = gp.Model("DCOPF")
# === Sets ===
buses = mpc_data['bus']['bus_i'].tolist()
buses_index_busID = dict(zip(mpc_data['bus'].index,mpc_data['bus']['bus_i']))
gens = mpc_data['gen']['gen_ID'].tolist()
branches = mpc_data['branch'].index.tolist()
branches_ftbus = dict(zip(branches,mpc_data['branch'][['bus_i', 'bus_j']].values))
# === Parameters ===
# Generator cost coefficients (all costs are incorporated)
c = {}
for i in mpc_data['gencost']["gen_ID"]:
    c[i] = [mpc_data['gencost']['c2'][i-1],mpc_data['gencost']['c1'][i-1],mpc_data['gencost']['c0'][i-1]]
# Bus power demand (MW)
Pd = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Pd']))
# Generator capacity limits (MW)
Pmax = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmax']))    
Pmin = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmin']))
# Transmission line limits (MW)
Pmax_line = dict(zip(branches, mpc_data['branch']['rateA']))
Pmin_line = dict(zip(branches, -mpc_data['branch']['rateA']))
# Line susceptance (1/X), assuming per unit values
B = dict(zip(branches, mpc_data['branch']['x']/(mpc_data['branch']['x']**2+mpc_data['branch']['r']**2)))
# Generator bus assignment
gen_bus = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['bus_i']))
# === Variables ===
Pg = model.addVars(gens, lb=Pmin, ub=Pmax, vtype=GRB.CONTINUOUS, name="Pg")
theta = model.addVars(buses, lb=-100, ub=100, vtype=GRB.CONTINUOUS, name="theta")
P_flow = model.addVars(branches, lb=Pmin_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="P_flow")
# === Objective Function (Minimize Generation Cost, all costs are incorporated) ===
model.setObjective(gp.quicksum(c[i][0]*Pg[i] + c[i][1]*Pg[i] + c[i][2] for i in gens), GRB.MINIMIZE)
# === Power Balance Constraints ===
for b in buses:
    expr = (gp.quicksum(Pg[i] for i in gen_bus if gen_bus[i] == b)
            + gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[1] == b)
            - gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[0] == b))
    model.addConstr(expr == Pd[b], name=f"power_balance_{b}")
# === Line Flow Constraints (DC Power Flow) ===
for l in branches:
    model.addConstr(P_flow[l] == B[l]*(theta[branches_ftbus[l][0]] - theta[branches_ftbus[l][1]]), name=f"line_flow_{l}")
# === Reference Bus Constraint (Slack Bus) ===
ref_bus_index = mpc_data['bus'][mpc_data['bus']['type'] == 3].index[0]
model.addConstr(theta[buses_index_busID[ref_bus_index]] == 0, name="theta_ref") # Bus 1 is the reference bus
# === Solve Model Using Gurobi ===
model.Params.OptimalityTol = 1e-8 # Higher precision
# model.setParam('OutputFlag', 0) # suppress the output
model.optimize() 

Set parameter OptimalityTol to value 1e-08
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
OptimalityTol  1e-08

Optimize a model with 4548 rows, 4938 columns and 13372 nonzeros
Model fingerprint: 0xb2d025d4
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [7e+00, 2e+02]
  Bounds range     [8e-01, 3e+05]
  RHS range        [1e-01, 8e+02]
Presolve removed 2337 rows and 2548 columns
Presolve time: 0.03s
Presolved: 2211 rows, 2390 columns, 8018 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -4.5408648e+07   4.194492e+05   0.000000e+00      0s
    1174    2.3076604e+06   0.000000e+00   0.000000e+00      0s

Solved in 1174 iterations and 0.22 seconds (0.13 work units)
Optimal objective  2.307660417e+06


,bus_i,type,Pd,Qd,Gs,Bs,area,Vm,Va,baseKV,zone,Vmax,Vmin
0,1,3,110,40,0,0,1,1,0,240,1,1.1,0.9
1,2,2,110,40,0,0,1,1,0,240,1,1.1,0.9
2,3,2,95,50,0,0,1,1,0,240,1,1.1,0.9


In [ ]:
# print("\nOptimal Generator Outputs (MW):")
# for g in gens:
#     print(f"Generator {g}: {Pg[g].X} MW")

# print("\nOptimal Line Flows (MW):")
# for l in branches:
#     print(f"Line {l}: {P_flow[l].X} MW")

# print("\nVoltage Angles (Radians):")
# for b in buses:
#     print(f"Bus {b}: {theta[b].X} rad")